# Telescope Runs — Stage 2 Demo (classical run ingest)

This notebook demonstrates `solsys_code/management/commands/load_telescope_runs.py`
(issue #37 Stage 2), the classical-schedule ingest command that parses run lines
and idempotently creates one `CalendarEvent` per observing night.

It demonstrates:

- Seeding `Observatory` records for the 4 supported sites (idempotent `update_or_create`)
- Writing a small sample schedule file to a temporary path
- Invoking `load_telescope_runs` via `call_command`
- Inspecting the resulting `CalendarEvent` rows (title, start/end times, description)
- Re-running the command to confirm idempotency (no duplicate events)

This notebook lives in `pre_executed/` because it is **DB-dependent** (it seeds
`Observatory` records and creates `CalendarEvent` rows) and is therefore **NOT
run during Sphinx/CI/ReadTheDocs builds**, per `docs/notebooks/README.md`.

## Django setup

Standard boilerplate to make `src.fomo.settings` importable from this notebook's
location (`docs/notebooks/pre_executed/` — three levels under the repo root, so
`parents[2]` gives the repo root) and to allow synchronous ORM calls inside
Jupyter's async event loop.

In [ ]:
import os
import sys
from pathlib import Path

import django

# Ensure the repo root is on sys.path so `src.fomo.settings` is importable
# when this notebook is executed from docs/notebooks/pre_executed/.
repo_root = str(Path.cwd().resolve().parents[2])
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'src.fomo.settings')

# Jupyter's ipykernel runs inside an asyncio event loop, but Django's ORM is
# sync-only by default and refuses to run there; this opts back in.
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

django.setup()

## Seed Observatory records

`load_telescope_runs` resolves telescope names to `Observatory` rows via MPC obscode
(through `telescope_runs.get_site()`). The 4 sites below match the values used in
`solsys_code/tests/test_telescope_runs.py`'s `setUpTestData`. Using
`update_or_create` makes this cell idempotent — safe to re-run against any dev DB.

In [ ]:
from solsys_code.solsys_code_observatory.models import Observatory

SITE_DATA = {
    '268': dict(
        name='Magellan Clay Telescope',
        short_name='Magellan-Clay',
        lat=-29.0146,
        lon=-70.6926,
        altitude=2402,
        timezone='America/Santiago',
    ),
    '269': dict(
        name='Magellan Baade Telescope',
        short_name='Magellan-Baade',
        lat=-29.0146,
        lon=-70.6926,
        altitude=2402,
        timezone='America/Santiago',
    ),
    '809': dict(
        name='ESO, La Silla',
        short_name='NTT',
        lat=-29.2567,
        lon=-70.7300,
        altitude=2347,
        timezone='America/Santiago',
    ),
    'E10': dict(
        name='Siding Spring Observatory',
        short_name='FTS',
        lat=-31.2734,
        lon=149.0612,
        altitude=1149,
        timezone='Australia/Sydney',
    ),
}

for obscode, fields in SITE_DATA.items():
    obs, created = Observatory.objects.update_or_create(obscode=obscode, defaults=fields)
    action = 'created' if created else 'updated/unchanged'
    print(f'  obscode={obscode!r:>4}  {action:>18}  short_name={obs.short_name!r}')

## Write a sample schedule file

A real schedule file is a plain-text file with one run line per non-blank line.
Here we write a small sample to a temporary file. The format accepted by
`parse_run_line` is flexible: the month name may appear before or after the
day range, and instrument names may be hyphenated.

In [ ]:
import tempfile

SAMPLE_SCHEDULE = """\
NTT EFOSC2 allocation 9-13 July
FTS MUSCAT4 allocation 10-12 July
Magellan LDSS3 14-16 July (proposed)
"""

with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as tmp:
    tmp.write(SAMPLE_SCHEDULE)
    schedule_path = tmp.name

print('Schedule file written to:', schedule_path)
print()
print('Contents:')
print(SAMPLE_SCHEDULE)

## Invoke the load_telescope_runs command

`call_command` is the Django-recommended way to invoke management commands
programmatically. The command will:

1. Parse each run line with `parse_run_line`
2. Resolve the telescope name to its `Observatory` via `get_site`
3. Expand the date range to one evening per night
4. For each night: compute `sun_event` sunset/sunrise and the -15° dark window
5. Upsert a `CalendarEvent` keyed on `(telescope, instrument, start_time)`

The end-of-run summary reports `created`, `updated`, `unchanged`, and `skipped` counts.

In [ ]:
import io

from django.core.management import call_command

stdout_buf = io.StringIO()
stderr_buf = io.StringIO()

call_command('load_telescope_runs', schedule_path, stdout=stdout_buf, stderr=stderr_buf)

print('stdout:', stdout_buf.getvalue())
if stderr_buf.getvalue():
    print('stderr (skipped lines):', stderr_buf.getvalue())

## Inspect the created CalendarEvent rows

Each observing night in the schedule produces one `CalendarEvent`. The `description`
field (D-06) contains the -15° dark-window UTC times, the run status, and the
original source line for traceability.

In [ ]:
from tom_calendar.models import CalendarEvent

events = CalendarEvent.objects.order_by('start_time')
print(f'Total CalendarEvent rows: {events.count()}')
print()

for ev in events:
    print(f'Title      : {ev.title}')
    print(f'start_time : {ev.start_time.isoformat()}')
    print(f'end_time   : {ev.end_time.isoformat()}')
    print('Description:')
    for line in ev.description.splitlines():
        print(f'  {line}')
    print()

## Idempotency check

Running the command a second time with the same file should produce zero new
events and zero updates — the summary should report `created: 0, updated: 0`.
This satisfies INGEST-03.

In [ ]:
stdout_buf2 = io.StringIO()
stderr_buf2 = io.StringIO()

call_command('load_telescope_runs', schedule_path, stdout=stdout_buf2, stderr=stderr_buf2)

print('Second run stdout:', stdout_buf2.getvalue())
count_after = CalendarEvent.objects.count()
print(f'CalendarEvent count unchanged: {count_after}')

## Summary

This notebook demonstrates the Phase 03 `load_telescope_runs` command satisfying
requirements INGEST-01, INGEST-02, and INGEST-03:

| Requirement | Description | Demonstrated by |
|-------------|-------------|------------------|
| INGEST-01 | One `CalendarEvent` per observing night, `start_time=sunset`, `end_time=sunrise` | 5 NTT + 3 FTS + 3 Magellan events created |
| INGEST-02 | `title`, `description` (dark window, status, source line), `telescope`, `instrument` populated | `ev.description` printed above shows all three D-06 lines |
| INGEST-03 | Idempotent re-run leaves row count and field values unchanged | Second run reports `created: 0, updated: 0` |

The command is invoked via `call_command` — equivalent to running
`./manage.py load_telescope_runs <filepath>` from the shell.

This notebook is **pre-executed** and intentionally excluded from automated doc
builds (not referenced in `docs/notebooks.rst`) because it depends on
`Observatory` records and live `CalendarEvent` writes in the local dev DB.